In [1]:

import sys, os
from glob import glob
from pathlib import Path
from pyspark.sql import functions as F

In [2]:
# Adiciona a pasta raiz do projeto (um ou dois níveis acima) no caminho do Python
sys.path.append(os.path.abspath(os.path.join('..')))  # Ajuste a quantidade de '..' conforme a profundidade da subpasta

# Cria uma conexão Spark 
from spark_utils import get_spark_session # ver em C:\Marco Conti\Projetos\MAIS-v2\spark_utils.py
spark = get_spark_session("MeuNotebook")

In [ ]:
# 1. Pastas de origem e destino
pasta_origem = Path(r"C:\Marco Conti\Projetos\Dados\DataSUS\SIH\parquet")
pasta_destino = Path(r"C:\Marco Conti\Projetos\Dados\DataSUS\SIH\csv")
pasta_destino.mkdir(exist_ok=True)

# 3. Busca todos os arquivos Parquet
arquivos_parquet = glob(str(pasta_origem / "*.parquet"))

arquivos_parquet_filtered = [arq for arq in arquivos_parquet if os.path.basename(arq)[4:6] == '08']
print(f"Convertendo {len(arquivos_parquet_filtered)} arquivos...")

for rec, arq in enumerate(arquivos_parquet_filtered):
    # 4. Lê o arquivo Parquet
    df_parquet = spark.read.parquet(arq)
    df_parquet = \
        (df_parquet.withColumns({"NOME_arquivo": F.lit(os.path.basename(arq))
                                ,"ANO_arquivo": F.concat_ws("20", F.lit(os.path.basename(arq)[4:6]))
                                ,"UF_arquivo": F.lit(os.path.basename(arq)[2:4])
                                }))
                   
    if rec == 0:
        df_parquet_acum = df_parquet
    else:
        df_parquet_acum = df_parquet_acum.union(df_parquet)

    

In [ ]:
save_path = r"C:\Marco Conti\Projetos\Dados\DataSUS\SIH\csv"
file_name = "teste_mconti.csv"
df_parquet_acum.toPandas().to_csv(f"{save_path}\{file_name}", index=False)


In [ ]:
df_parquet_acum.select("ANO_arquivo", "UF_arquivo").groupBy("ANO_arquivo", "UF_arquivo").count().show(30, truncate=False)

In [9]:
import duckdb
from pathlib import Path
import zipfile

# Diretórios de entrada e saída

pasta_parquet = Path(r"C:\Marco Conti\Projetos\Dados\DataSUS\SIH\parquet")
pasta_saida = Path(r"C:\Marco Conti\Projetos\Dados\DataSUS\SIH\csv")
pasta_saida.mkdir(parents=True, exist_ok=True)

# Inicializa conexão em memória
con = duckdb.connect()

# # 1. Mapeia todos os arquivos Parquet e extrai os anos disponíveis
# # O padrão 'RD..(\d{2})\d{2}\.parquet' extrai os 2 dígitos do ano
# query_anos = f"""
#     SELECT DISTINCT 
#         regexp_extract(filename, 'RD..(\\d{{2}})\\d{{2}}\\.\\w+$', 1) AS ano_2dig
#     FROM read_parquet('{pasta_parquet}/*.parquet', filename=TRUE)
#     WHERE regexp_extract(filename, 'RD..(\\d{{2}})\\d{{2}}\\.\\w+$', 1) != ''
# """

# anos = [row[0] for row in con.execute(query_anos).fetchall()]

anos = [20, 21, 22, 23, 24, 25]

print(f"Anos encontrados nos arquivos: {sorted(anos)}")

# 2. Processa e gera 1 CSV para cada ano
for ano in sorted(anos):
    # Formata o ano para 4 dígitos (ex: '08' -> '2008')
    ano_4dig = f"20{ano}" if int(ano) < 50 else f"19{ano}"
    
    nome_base = f"RD_BR_{ano_4dig}"
    arquivo_csv = pasta_saida / f"RD_BR_{ano_4dig}.csv"
    arquivo_zip = pasta_saida / f"{nome_base}.zip"
    print(f"Gerando {arquivo_csv.name}...")
    
    # Seleciona todos os arquivos correspondentes àquele ano (* -> UF, {ano} -> Ano, ?? -> Mês)
    con.execute(f"""
        COPY (
            SELECT * 
            FROM read_parquet('{pasta_parquet}/RD??{ano}??.parquet')
        ) TO '{arquivo_csv}' (HEADER, DELIMITER ';');
    """)

    print(f"2/3. Compactando para ZIP: {arquivo_zip.name}...")
    # Criar o arquivo .zip contendo o .csv
    with zipfile.ZipFile(arquivo_zip, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=9) as zipf:
        # arcname garante que dentro do zip fique apenas o arquivo, sem a estrutura de pastas do sistema
        zipf.write(arquivo_csv, arcname=arquivo_csv.name)
        
    print(f"3/3. Removendo arquivo CSV temporário...")
    arquivo_csv.unlink()    

con.close()
print("Conversão concluída com sucesso!")

Anos encontrados nos arquivos: [20, 21, 22, 23, 24, 25]
Gerando RD_BR_2020.csv...
2/3. Compactando para ZIP: RD_BR_2020.zip...
3/3. Removendo arquivo CSV temporário...
Gerando RD_BR_2021.csv...
2/3. Compactando para ZIP: RD_BR_2021.zip...
3/3. Removendo arquivo CSV temporário...
Gerando RD_BR_2022.csv...
2/3. Compactando para ZIP: RD_BR_2022.zip...
3/3. Removendo arquivo CSV temporário...
Gerando RD_BR_2023.csv...
2/3. Compactando para ZIP: RD_BR_2023.zip...
3/3. Removendo arquivo CSV temporário...
Gerando RD_BR_2024.csv...
2/3. Compactando para ZIP: RD_BR_2024.zip...
3/3. Removendo arquivo CSV temporário...
Gerando RD_BR_2025.csv...
2/3. Compactando para ZIP: RD_BR_2025.zip...
3/3. Removendo arquivo CSV temporário...
Conversão concluída com sucesso!
